# 📧 Spam Email Detector
A deep learning classifier that detects spam SMS messages using an LSTM neural network.

**Run all cells in order:** Runtime → Run all (`Ctrl+F9`)

## Step 1 — Install & Import Libraries

In [ ]:
!pip install tensorflow pandas scikit-learn seaborn gradio --quiet

import pandas as pd
import numpy as np
import tensorflow as tf
import seaborn as sns
import matplotlib.pyplot as plt

from tensorflow.keras.models import Sequential, load_model
# Notice the upgraded layers: TextVectorization, Bidirectional, and GlobalMaxPool1D
from tensorflow.keras.layers import TextVectorization, Embedding, Bidirectional, LSTM, GlobalMaxPool1D, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

print('✅ All modern deep learning libraries imported successfully!')
print(f'   TensorFlow version: {tf.__version__}')

## Step 2 — Load the Dataset

In [ ]:
url = 'https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv'
df = pd.read_csv(url, sep='\t', header=None, names=['label', 'message'])

print('📊 Dataset Overview')
print('=' * 40)
print(df.head())
print(f'\nTotal messages : {len(df)}')
print(f'Spam           : {(df["label"] == "spam").sum()} ({(df["label"] == "spam").mean():.1%})')
print(f'Ham            : {(df["label"] == "ham").sum()} ({(df["label"] == "ham").mean():.1%})')

## Step 3 — Preprocess the Text

In [ ]:
# --- Configuration Constants ---
MAX_WORDS  = 10000
MAX_LENGTH = 150  # Great job increasing this to 150 for more context!

# Encode labels: spam=1, ham=0
df['label_enc'] = (df['label'] == 'spam').astype(int)

# Train / test split using RAW text strings (not padded numbers)
X_train, X_test, y_train, y_test = train_test_split(
    df['message'].values,         # .values converts this into clean text arrays
    df['label_enc'].values,
    test_size=0.2, 
    random_state=42, 
    stratify=df['label_enc']
)

# Class weights to handle imbalance (87% ham vs 13% spam)
weights       = compute_class_weight('balanced', classes=np.array([0, 1]), y=y_train)
class_weights = {0: weights[0], 1: weights[1]}

print('✅ Splitting and structural balancing complete!')
print(f'   Train samples : {len(X_train)}')
print(f'   Test samples  : {len(X_test)}')
print(f'   Class weights : Ham={class_weights[0]:.2f}, Spam={class_weights[1]:.2f}')

## Step 4 — Build the Model (LSTM)

In [ ]:
# Initialize the modern Text Vectorization Layer
vectorizer_layer = TextVectorization(
    max_tokens=MAX_WORDS,
    output_sequence_length=MAX_LENGTH,
    output_mode='int'
)

# Teach the vectorizer all the words in our training data
vectorizer_layer.adapt(X_train)

# Construct the upgraded, self-contained AI architecture
model = Sequential([
    # 1. Input Layer: Tells the model to accept raw text strings directly
    tf.keras.Input(shape=(1,), dtype=tf.string, name="raw_text_input"),
    
    # 2. Vectorization: Automatically converts text to numbers
    vectorizer_layer,
    
    # 3. Embedding: Turns numbers into 128-dimensional meaning vectors (mask_zero ignores padding)
    Embedding(input_dim=MAX_WORDS, output_dim=128, mask_zero=True, name="masked_embedding"),
    
    # 4. Bidirectional LSTM: Reads sentences forward AND backward for full context
    Bidirectional(LSTM(64, return_sequences=True), name="bidirectional_lstm"),
    
    # 5. Global Max Pooling: Squeezes the data down, grabbing the most intense spam indicators
    GlobalMaxPool1D(name="global_max_pooling"),
    
    # 6. Dropout & Dense Layers: Regularizes learning to prevent memorization/cheating
    Dropout(0.4, name="dropout_1"),
    Dense(32, activation='relu', name="dense_feature_extractor"),
    Dropout(0.2, name="dropout_2"),
    
    # 7. Output Node: Gives a final probability score between 0.0 (Ham) and 1.0 (Spam)
    Dense(1, activation='sigmoid', name="output_prediction_node")
])

# Compile with advanced tracking metrics
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.Precision(name='precision'), tf.keras.metrics.Recall(name='recall')]
)

# Show the new, upgraded blueprints
model.summary()

## Step 5 — Train the Model

In [ ]:
# 1. Set up the callbacks (The AI's safety guardrails)
early_stop = EarlyStopping(
    monitor='val_loss', 
    patience=3, 
    restore_best_weights=True, 
    verbose=1
)

# NEW: Saves the actual model file to your disk ONLY when it beats its previous best score
checkpoint = ModelCheckpoint(
    'spam_detector_best.keras', 
    monitor='val_loss', 
    save_best_only=True, 
    verbose=1
)

# 2. Feed the raw text directly into the training loop!
history = model.fit(
    X_train, y_train,              # Swapped out X_train_pad for clean raw text strings!
    epochs=15,                     # 15 epochs is plenty for this smarter architecture
    batch_size=64,                 # Increased batch size to 64 for faster, more stable learning
    validation_split=0.1,          # Holds back 10% of training data to test itself during training
    class_weight=class_weights,    # Tells the AI to pay extra attention to the rare Spam messages
    callbacks=[early_stop, checkpoint], # Added our new checkpoint saver here
    verbose=1
)

print(f'\n✅ Training pipeline ended at epoch {len(history.history["loss"])}')

## Step 6 — Evaluate the Model

In [ ]:
# 1. Test dataset evaluation using raw text strings
metrics_eval = model.evaluate(X_test, y_test, verbose=0)
print(f'Test Accuracy  : {metrics_eval[1]:.4f}')
print(f'Test Precision : {metrics_eval[2]:.4f}')
print(f'Test Recall    : {metrics_eval[3]:.4f}\n')

# 2. Detailed Classification Report
y_pred = (model.predict(X_test, verbose=0) > 0.5).astype(int)
print(classification_report(y_test, y_pred, target_names=['Ham', 'Spam']))

# 3. Plot Training History Trajectories
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history['accuracy'],     label='Train')
axes[0].plot(history.history['val_accuracy'], label='Validation')
axes[0].set_title('Accuracy Trajectory')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(history.history['loss'],     label='Train')
axes[1].plot(history.history['val_loss'], label='Validation')
axes[1].set_title('Loss Trajectory')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.tight_layout()
plt.show()

# 4. Heatmap Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 3.5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Ham', 'Spam'], 
            yticklabels=['Ham', 'Spam'])
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix Visualization')
plt.show()

## Step 7 — Save the Model

In [ ]:
# 1. Save your completed, end-to-end model brain
model.save('spam_detector.keras')
print('✅ Model successfully saved to your disk as spam_detector.keras')

# 2. NEW: Let's immediately reload it into a fresh variable to prove it works independently!
reloaded_model = load_model('spam_detector.keras')
print('✅ Self-contained reloaded_model is loaded and ready for action!')

## Step 8 — Test with Your Own Messages

In [ ]:
def predict_spam(text):
    # Pass the raw text string directly to the model wrapped in a list [text]
    # No more manual tokenizing or padding required!
    prob = reloaded_model.predict([text], verbose=0)[0][0]
    
    label = 'SPAM' if prob > 0.5 else 'HAM'
    print(f'Message : {text[:70]}')
    print(f'Result  : {label}  (confidence: {prob:.2%})\n')

# Test your newly upgraded AI on custom sentences!
predict_spam("Congratulations! You've won a FREE iPhone. Click here now!")
predict_spam("Hey, are we still meeting for lunch tomorrow?")
predict_spam("URGENT: Your account will be suspended. Verify NOW: bit.ly/x")
predict_spam("Can you send me the lecture slides from today?")
predict_spam("You have been selected for a cash prize of $1000. Call us now!")

## Step 9 — Interactive Web App (Gradio)

In [ ]:
import gradio as gr

def check_spam(text):
    # If the user clicks 'Submit' without typing anything, stop early
    if not text.strip():
        return 'Please enter a message.'
        
    # Use the self-contained reloaded model with the raw text input string!
    prob = reloaded_model.predict([text], verbose=0)[0][0]
    
    label = 'SPAM' if prob > 0.5 else 'HAM'
    return f'{label} — confidence: {prob:.2%}'

# Launch the interactive Web App UI
gr.Interface(
    fn=check_spam,
    inputs=gr.Textbox(lines=3, placeholder='Type a message to check...', label='Message'),
    outputs=gr.Textbox(label='Result'),
    title='📧 Upgraded Spam Detector Web App',
    description='Enter any text or message block to run real-time deep learning inference predictions.',
    examples=[
        ['Congratulations! You won a FREE prize. Call now!'],
        ['Are you coming to class tomorrow?'],
        ['URGENT: Your bank account is locked. Verify identity at bit.ly/fakebank']
    ]
).launch()